# Análisis exploratorio (EDA)

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.utils.config import cargar_config, ruta_csv, carpeta
from src.ingesta import leer_csv
from src.limpieza import limpiar_ejercicio
cfg = cargar_config()
SAL = carpeta(cfg, 'salidas')

## 1. Conteos por ejercicio y control contra lo esperado

In [ ]:
plata, invalidos = [], []
for a in cfg['proyecto']['ejercicios']:
    out, inv, st = limpiar_ejercicio(leer_csv(ruta_csv(cfg, a), cfg), cfg)
    plata.append(out); invalidos.append(inv.assign(ejercicio=a))
df = pd.concat(plata, ignore_index=True); inv = pd.concat(invalidos, ignore_index=True)
conteo = df.groupby('anio').size().rename('registros').to_frame()
conteo['esperado'] = pd.Series(cfg['fuente']['registros_esperados'])
conteo['ok'] = conteo.registros == conteo.esperado
conteo

## 2. Valores inválidos y nulos

In [ ]:
print(inv.groupby(['columna','motivo']).size())
imp = [c for c in df.columns if str(df[c].dtype) == 'float64']
nulos = df[imp].isna().mean().mul(100).round(2).sort_values(ascending=False)
nulos.head(15)

## 3. Distribución de los importes (escala logarítmica)

In [ ]:
df['variacion'] = df['total_bienes_final'] - df['total_bienes_inicio']
cols = ['total_bienes_final','total_deudas_final','total_ingreso_neto_c1234','variacion']
fig, ax = plt.subplots(2, 2, figsize=(11, 7))
for a_, c in zip(ax.ravel(), cols):
    s = df[c].dropna(); s = s[s > 0]
    a_.hist(np.log10(s), bins=60); a_.set_title(f'log10({c}) | n={len(s):,}')
plt.tight_layout(); plt.savefig(SAL / 'eda_histogramas.png', dpi=150)

## 4. Casos especiales ya identificados

In [ ]:
an = df[df.es_analizada]
print('Anuales analizadas:', len(an))
print('Ingreso neto igual a cero (%):', round((df.total_ingreso_neto_c1234 == 0).mean()*100, 1))
print('Bienes finales <= 0 (%):', round((df.total_bienes_final <= 0).mean()*100, 1))
print('diferencia_valuacion con negativos:', int((df.diferencia_valuacion < 0).sum()))
print('Funcionarios distintos:', df.cuit.nunique())

## 5. Cardinalidad y calidad de las dimensiones

In [ ]:
print('organismos:', df.organismo.nunique(), '| ámbitos:', df.actividad_principal_ambito.nunique())
print('cargos crudos:', df.cargo.nunique(), '| normalizados:', df.cargo_normalizado.nunique())
print(df.sector_normalizado.value_counts())
print(df.tipo_dj.value_counts())

## 6. Funcionarios por cantidad de ejercicios

In [ ]:
n = df.groupby('cuit').anio.nunique()
print('1 ejercicio (%):', round((n == 1).mean()*100, 1), '| 2 o más (%):', round((n >= 2).mean()*100, 1), '| mediana:', n.median(), '| 10 ejercicios:', int((n == 10).sum()))

## 7. Coherencia entre ejercicios
Compara los bienes iniciales del año *t* con los bienes finales del año *t−1* del mismo CUIT (solo declaraciones analizadas).

In [ ]:
a = an[['cuit','anio','total_bienes_inicio','total_bienes_final']].dropna()
prev = a.assign(anio=a.anio + 1)[['cuit','anio','total_bienes_final']].rename(columns={'total_bienes_final':'final_previo'})
m = a.merge(prev, on=['cuit','anio'])
print('pares comparables:', len(m), '| coinciden (%):', round((np.isclose(m.total_bienes_inicio, m.final_previo, atol=1)).mean()*100, 1))

## 8. Conclusiones
Escribí aquí un párrafo por apartado: calidad de datos, casos especiales, cardinalidades, cobertura longitudinal y coherencia entre ejercicios.